# CS1 EXP-2-IDABS — SVD(TF-IDF) + Static Features → MLP

This notebook runs the **identifier-abstraction variant** of EXP-2.

It keeps the EXP-2 model architecture unchanged and changes only the code representation:

```text
abstracted_code_v1
→ word TF-IDF + character TF-IDF
→ train-fold-only TruncatedSVD
+ deterministic static source-level features
→ shallow MLP classifier
→ project-grouped 5-fold development OOF evaluation
```

## Scope

This notebook is **development-only**.

The frozen 20% outer holdout is loaded only for split verification. It is **not evaluated** here because the final holdout has already been used for the locked selected EXP-2 normalized-code model.


## 0. What this notebook fixes

The draft `03_cs1_exp2_mlp.ipynb` mixed EXP-2 logic with IDABS assumptions. This version fixes the important issues:

1. Loads `rdiversevul_cs1_normalized_plus_abstracted_v1.parquet`, not the normalized-only parquet.
2. Uses `abstracted_code_v1` explicitly as the MLP text input.
3. Keeps artifacts in a separate `exp2_mlp_idabs_dev_cv_*` output folder.
4. Repairs rare empty abstracted-code rows using a deterministic sentinel token.
5. Reuses the frozen 80/20 project-disjoint outer split and frozen 5-fold project-grouped inner CV manifest.
6. Uses development CV only; final outer holdout evaluation is blocked.
7. Provides profile mode first, then official 5-fold development CV.


## 1. Mount Google Drive


In [2]:
from google.colab import drive

drive.mount("/content/drive")


Mounted at /content/drive


## 2. Runtime switches

Run profile first. After the profile succeeds, disable profile and enable the official development CV.


In [3]:
# ============================================================================
# Runtime switches
# ============================================================================

# First run with profile only. It checks that the full pipeline works on one fold.
RUN_PROFILE_FOLD = False

# Set True only after profile mode succeeds.
RUN_OFFICIAL_DEV_CV = True

# Load saved official artifacts when they already exist.
LOAD_EXISTING_DEV_CV_ARTIFACTS = True

# Keep final outer holdout locked. Do not enable for this post-hoc IDABS study.
RUN_FINAL_OUTER_HOLDOUT = False

# Official full CV should run on GPU when possible.
REQUIRE_GPU_FOR_OFFICIAL_RUN = True

# Reproducibility.
REPO_URL = "https://github.com/EnomisLP/DiverseVul--IS-Project.git"
REPO_BRANCH = "prashant"
INNER_N_SPLITS = 5
MODEL_RANDOM_STATE = 42

# Increment only when intentionally creating a clean new artifact folder.
RUN_VERSION = "v1"

print("RUN_PROFILE_FOLD:", RUN_PROFILE_FOLD)
print("RUN_OFFICIAL_DEV_CV:", RUN_OFFICIAL_DEV_CV)
print("LOAD_EXISTING_DEV_CV_ARTIFACTS:", LOAD_EXISTING_DEV_CV_ARTIFACTS)
print("RUN_FINAL_OUTER_HOLDOUT:", RUN_FINAL_OUTER_HOLDOUT)
print("RUN_VERSION:", RUN_VERSION)


RUN_PROFILE_FOLD: False
RUN_OFFICIAL_DEV_CV: True
LOAD_EXISTING_DEV_CV_ARTIFACTS: True
RUN_FINAL_OUTER_HOLDOUT: False
RUN_VERSION: v1


## 3. Define paths


In [4]:
from pathlib import Path

DRIVE_ROOT = Path(
    "/content/drive/MyDrive/IntelligentSystemProject/VulnerabilityDetectionData"
)

PROCESSED_DIR = DRIVE_ROOT / "processed"
MANIFEST_ROOT = DRIVE_ROOT / "manifests"
OUTPUT_ROOT = DRIVE_ROOT / "outputs"

EXPERIMENT_ID = "cs1_project_holdout20_innercv_v1"
EXPERIMENT_MANIFEST_DIR = MANIFEST_ROOT / EXPERIMENT_ID
EXPERIMENT_OUTPUT_DIR = OUTPUT_ROOT / EXPERIMENT_ID

OUTER_SPLIT_DIR = EXPERIMENT_MANIFEST_DIR / "outer_holdout"
INNER_SPLIT_DIR = EXPERIMENT_MANIFEST_DIR / "inner_cv"
STATIC_FEATURE_DIR = PROCESSED_DIR / "static_features"

NORMALIZED_DATA_PATH = PROCESSED_DIR / "rdiversevul_cs1_normalized_v1.parquet"
ABSTRACTED_DATA_PATH = PROCESSED_DIR / "rdiversevul_cs1_normalized_plus_abstracted_v1.parquet"
OUTER_MANIFEST_PATH = OUTER_SPLIT_DIR / "cs1_outer_project_holdout_manifest.parquet"
INNER_MANIFEST_PATH = INNER_SPLIT_DIR / "cs1_project_grouped_5fold_manifest.parquet"
INNER_SELECTION_METADATA_PATH = (
    INNER_SPLIT_DIR / "cs1_inner_grouped_split_selection_metadata.json"
)
STATIC_FEATURE_PATH = STATIC_FEATURE_DIR / "cs1_static_features_v1.parquet"

EXP2_IDABS_OUTPUT_DIR = EXPERIMENT_OUTPUT_DIR / f"exp2_mlp_idabs_dev_cv_{RUN_VERSION}"
EXP2_IDABS_PROFILE_DIR = EXP2_IDABS_OUTPUT_DIR / "profile_fold"

for directory in [
    PROCESSED_DIR,
    MANIFEST_ROOT,
    OUTPUT_ROOT,
    EXPERIMENT_MANIFEST_DIR,
    EXPERIMENT_OUTPUT_DIR,
    EXP2_IDABS_OUTPUT_DIR,
    EXP2_IDABS_PROFILE_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

print("Abstracted dataset:", ABSTRACTED_DATA_PATH)
print("Frozen outer manifest:", OUTER_MANIFEST_PATH)
print("Frozen inner manifest:", INNER_MANIFEST_PATH)
print("Static feature cache:", STATIC_FEATURE_PATH)
print("EXP-2-IDABS output:", EXP2_IDABS_OUTPUT_DIR)


Abstracted dataset: /content/drive/MyDrive/IntelligentSystemProject/VulnerabilityDetectionData/processed/rdiversevul_cs1_normalized_plus_abstracted_v1.parquet
Frozen outer manifest: /content/drive/MyDrive/IntelligentSystemProject/VulnerabilityDetectionData/manifests/cs1_project_holdout20_innercv_v1/outer_holdout/cs1_outer_project_holdout_manifest.parquet
Frozen inner manifest: /content/drive/MyDrive/IntelligentSystemProject/VulnerabilityDetectionData/manifests/cs1_project_holdout20_innercv_v1/inner_cv/cs1_project_grouped_5fold_manifest.parquet
Static feature cache: /content/drive/MyDrive/IntelligentSystemProject/VulnerabilityDetectionData/processed/static_features/cs1_static_features_v1.parquet
EXP-2-IDABS output: /content/drive/MyDrive/IntelligentSystemProject/VulnerabilityDetectionData/outputs/cs1_project_holdout20_innercv_v1/exp2_mlp_idabs_dev_cv_v1


## 4. Clone or refresh the repository branch


In [5]:
from pathlib import Path
import sys
import subprocess

REPO_DIR = Path("/content/DiverseVul--IS-Project")
PROJECT_DIR = REPO_DIR / "vuln-detection"
SRC_DIR = PROJECT_DIR / "src"

if not REPO_DIR.exists():
    subprocess.run(
        [
            "git",
            "clone",
            "--branch",
            REPO_BRANCH,
            "--single-branch",
            REPO_URL,
            str(REPO_DIR),
        ],
        check=True,
    )
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", REPO_BRANCH], check=True)

if not SRC_DIR.exists():
    raise FileNotFoundError(f"Expected source directory does not exist: {SRC_DIR}")

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

repo_commit = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"],
    text=True,
).strip()

print("Repository:", REPO_DIR)
print("Source path:", SRC_DIR)
print("Commit:", repo_commit)


Repository: /content/DiverseVul--IS-Project
Source path: /content/DiverseVul--IS-Project/vuln-detection/src
Commit: 657ce09edcc1896d83bf1e68f2f11e714382907b


## 5. Verify required repository files


In [6]:
required_repo_files = [
    SRC_DIR / "case_study_1" / "evaluation.py",
    SRC_DIR / "case_study_1" / "split_manifest.py",
    SRC_DIR / "case_study_1" / "exp1" / "static_features.py",
    SRC_DIR / "case_study_1" / "exp2" / "__init__.py",
    SRC_DIR / "case_study_1" / "exp2" / "exp2_mlp.py",
]

missing_repo_files = [str(path) for path in required_repo_files if not path.exists()]
if missing_repo_files:
    raise FileNotFoundError(
        "Required EXP-2 files are missing:\n" + "\n".join(missing_repo_files)
    )

print("Required Case Study 1 files are present.")


Required Case Study 1 files are present.


## 6. Install dependencies


In [7]:
import sys
import subprocess

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "numpy",
        "pandas",
        "scipy",
        "scikit-learn",
        "matplotlib",
        "pyyaml",
        "pyarrow",
        "joblib",
    ],
    check=True,
)
print("Dependencies installed or already available.")


Dependencies installed or already available.


## 7. Import modules and check accelerator


In [8]:
import importlib
import json
import time

import numpy as np
import pandas as pd
import torch
from IPython.display import display

exp2_mlp = importlib.import_module("case_study_1.exp2.exp2_mlp")
static_features = importlib.import_module("case_study_1.exp1.static_features")
split_manifest = importlib.import_module("case_study_1.split_manifest")
evaluation = importlib.import_module("case_study_1.evaluation")

required_exp2_api = [
    "EXP2_VERSION",
    "Exp2Config",
    "run_exp2_profile_fold",
    "run_exp2",
]
missing_exp2_api = [name for name in required_exp2_api if not hasattr(exp2_mlp, name)]
if missing_exp2_api:
    raise AttributeError(f"EXP-2 runner is missing API: {missing_exp2_api}")

print("EXP-2 runner:", exp2_mlp.__file__)
print("EXP-2 version:", exp2_mlp.EXP2_VERSION)
print("Static feature module:", static_features.__file__)
print("Static feature count:", len(static_features.FEATURE_COLUMNS))
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory (GiB):",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
    )
else:
    print(
        "GPU is not active. Profile can still run on CPU if needed, but the official "
        "full MLP CV should use a GPU. In Colab: Runtime > Change runtime type > GPU."
    )


EXP-2 runner: /content/DiverseVul--IS-Project/vuln-detection/src/case_study_1/exp2/exp2_mlp.py
EXP-2 version: cs1-exp2-svd-static-mlp-v1-inner-val-holdout
Static feature module: /content/DiverseVul--IS-Project/vuln-detection/src/case_study_1/exp1/static_features.py
Static feature count: 54
PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
GPU memory (GiB): 14.56


## 8. Load abstracted dataset, frozen manifests, and static features


In [10]:
for required_path in [
    ABSTRACTED_DATA_PATH,
    OUTER_MANIFEST_PATH,
    INNER_MANIFEST_PATH,
    STATIC_FEATURE_PATH,
]:
    if not required_path.exists():
        raise FileNotFoundError(
            f"Missing frozen artifact: {required_path}\n"
            "Restore the completed preprocessing/split artifacts. Do not recreate "
            "split manifests in this notebook."
        )

full_df = pd.read_parquet(ABSTRACTED_DATA_PATH)
outer_manifest_df = pd.read_parquet(OUTER_MANIFEST_PATH)
inner_manifest_df = pd.read_parquet(INNER_MANIFEST_PATH)
static_df = pd.read_parquet(STATIC_FEATURE_PATH)

required_full_columns = {
    "source_row_id",
    "normalized_code",
    "abstracted_code_v1",
    "label",
    "project",
}
missing_full_columns = required_full_columns.difference(full_df.columns)
if missing_full_columns:
    raise KeyError(
        "Abstracted dataset is missing required columns:\n"
        f"{sorted(missing_full_columns)}\n\n"
        f"Available columns:\n{sorted(full_df.columns.tolist())}"
    )

required_outer_columns = {"source_row_id", "label", "project", "partition"}
missing_outer_columns = required_outer_columns.difference(outer_manifest_df.columns)
if missing_outer_columns:
    raise KeyError(f"Outer manifest missing columns: {sorted(missing_outer_columns)}")

selected_inner_split_seed = 42
if INNER_SELECTION_METADATA_PATH.exists():
    with INNER_SELECTION_METADATA_PATH.open("r", encoding="utf-8") as file:
        selected_inner_split_seed = int(
            json.load(file).get("selected_split_seed", 42)
        )

selected_inner_split_config = split_manifest.SplitConfig(
    n_splits=INNER_N_SPLITS,
    random_state=selected_inner_split_seed,
    shuffle=True,
)
split_manifest.assert_manifest_integrity(
    inner_manifest_df,
    config=selected_inner_split_config,
)

print("Loaded abstracted dataset:", ABSTRACTED_DATA_PATH)
print("Full dataset shape:", full_df.shape)
print("Outer manifest shape:", outer_manifest_df.shape)
print("Inner manifest shape:", inner_manifest_df.shape)
print("Static feature shape:", static_df.shape)
print("Selected inner split seed:", selected_inner_split_config.random_state)
print("Dataset columns:", sorted(full_df.columns.tolist()))
print("\nOuter partitions:")
print(outer_manifest_df["partition"].value_counts(dropna=False))


Loaded abstracted dataset: /content/drive/MyDrive/IntelligentSystemProject/VulnerabilityDetectionData/processed/rdiversevul_cs1_normalized_plus_abstracted_v1.parquet
Full dataset shape: (261667, 6)
Outer manifest shape: (261667, 5)
Inner manifest shape: (203958, 4)
Static feature shape: (261667, 55)
Selected inner split seed: 674
Dataset columns: ['abstracted_code_v1', 'code', 'label', 'normalized_code', 'project', 'source_row_id']

Outer partitions:
partition
development      203958
outer_holdout     57709
Name: count, dtype: int64


## 9. Strict split and leakage validation


In [11]:
# ============================================================================
# Validate dataset/manifest alignment and build development-only frames.
# ============================================================================

for name, frame in [
    ("full_df", full_df),
    ("outer_manifest_df", outer_manifest_df),
    ("inner_manifest_df", inner_manifest_df),
]:
    if frame["source_row_id"].duplicated().any():
        raise RuntimeError(f"{name} contains duplicate source_row_id values.")

full_indexed = full_df.set_index("source_row_id", drop=False).sort_index()

outer_ids = set(outer_manifest_df["source_row_id"].tolist())
dataset_ids = set(full_indexed.index.tolist())

if outer_ids != dataset_ids:
    raise RuntimeError(
        "Outer manifest IDs must exactly match the abstracted dataset.\n"
        f"Missing from dataset: {len(outer_ids - dataset_ids):,}\n"
        f"Extra in dataset: {len(dataset_ids - outer_ids):,}"
    )

outer_join = outer_manifest_df.set_index("source_row_id").join(
    full_indexed[["label", "project"]],
    rsuffix="_dataset",
)

if not (outer_join["label"].astype(int) == outer_join["label_dataset"].astype(int)).all():
    raise RuntimeError("Label mismatch between abstracted dataset and outer manifest.")

if not (
    outer_join["project"].astype(str).str.strip()
    == outer_join["project_dataset"].astype(str).str.strip()
).all():
    raise RuntimeError("Project mismatch between abstracted dataset and outer manifest.")

observed_partitions = set(outer_manifest_df["partition"].dropna().astype(str))
expected_partitions = {"development", "outer_holdout"}
if observed_partitions != expected_partitions:
    raise RuntimeError(
        f"Unexpected outer manifest partitions: {observed_partitions}. "
        f"Expected exactly: {expected_partitions}."
    )

dev_ids = set(
    outer_manifest_df.loc[
        outer_manifest_df["partition"] == "development",
        "source_row_id",
    ].tolist()
)
holdout_ids = set(
    outer_manifest_df.loc[
        outer_manifest_df["partition"] == "outer_holdout",
        "source_row_id",
    ].tolist()
)
inner_ids = set(inner_manifest_df["source_row_id"].tolist())

if not dev_ids or not holdout_ids:
    raise RuntimeError("Development or outer_holdout partition is empty.")

if dev_ids.intersection(holdout_ids):
    raise RuntimeError("Development and outer-holdout ID overlap detected.")

if inner_ids != dev_ids:
    raise RuntimeError(
        "Inner CV manifest IDs must exactly equal development IDs.\n"
        f"Missing from inner manifest: {len(dev_ids - inner_ids):,}\n"
        f"Extra in inner manifest: {len(inner_ids - dev_ids):,}"
    )

dev_projects = set(
    outer_manifest_df.loc[
        outer_manifest_df["partition"] == "development",
        "project",
    ].astype(str).str.strip()
)
holdout_projects = set(
    outer_manifest_df.loc[
        outer_manifest_df["partition"] == "outer_holdout",
        "project",
    ].astype(str).str.strip()
)
project_overlap = dev_projects.intersection(holdout_projects)
if project_overlap:
    raise RuntimeError(
        "Development/holdout project overlap detected: "
        f"{sorted(project_overlap)[:10]}"
    )

# Deterministic row order.
development_ids_sorted = sorted(dev_ids)
holdout_ids_sorted = sorted(holdout_ids)

dev_df = full_indexed.loc[development_ids_sorted].copy().reset_index(drop=True)
holdout_df = full_indexed.loc[holdout_ids_sorted].copy().reset_index(drop=True)

if set(dev_df["source_row_id"]).intersection(holdout_ids):
    raise RuntimeError("Outer-holdout rows leaked into dev_df.")

inner_fold_summary = split_manifest.summarize_manifest(
    inner_manifest_df,
    config=selected_inner_split_config,
)
if not (inner_fold_summary["train_test_project_overlap"] == 0).all():
    raise RuntimeError("At least one inner fold has train/test project overlap.")
if not (inner_fold_summary["test_vulnerable"] > 0).all():
    raise RuntimeError("At least one inner fold has zero vulnerable test samples.")
if not (inner_fold_summary["test_non_vulnerable"] > 0).all():
    raise RuntimeError("At least one inner fold has zero non-vulnerable test samples.")

summary_rows = []
for name, ids in [
    ("full_manifest", sorted(outer_ids)),
    ("development", development_ids_sorted),
    ("outer_holdout_locked", holdout_ids_sorted),
]:
    temp = full_indexed.loc[ids]
    summary_rows.append(
        {
            "partition": name,
            "rows": int(len(temp)),
            "projects": int(temp["project"].astype(str).nunique()),
            "vulnerable": int(temp["label"].astype(int).sum()),
            "positive_rate": float(temp["label"].astype(int).mean()),
        }
    )

split_summary_df = pd.DataFrame(summary_rows)
display(split_summary_df)

print("Frozen split validation passed.")
print("Development rows used by EXP-2-IDABS:", f"{len(dev_df):,}")
print("Global outer-holdout rows excluded from training/CV:", f"{len(holdout_df):,}")
print("\nInner development fold summary:")
display(inner_fold_summary)


,partition,rows,projects,vulnerable,positive_rate
0,full_manifest,261667,797,13938,0.053266
1,development,203958,594,10727,0.052594
2,outer_holdout_locked,57709,203,3211,0.055641


Frozen split validation passed.
Development rows used by EXP-2-IDABS: 203,958
Global outer-holdout rows excluded from training/CV: 57,709

Inner development fold summary:


,fold,test_rows,test_vulnerable,test_non_vulnerable,test_positive_rate,positive_rate_delta_from_global,test_unique_projects,train_rows,train_unique_projects,train_test_project_overlap,test_row_share,test_project_share
0,0,55509,2420,53089,0.043597,-0.008998,1,148449,593,0,0.272159,0.001684
1,1,40791,2091,38700,0.051261,-0.001333,158,163167,436,0,0.199997,0.265993
2,2,39072,2062,37010,0.052774,0.000180,146,164886,448,0,0.191569,0.245791
3,3,35016,2047,32969,0.058459,0.005865,137,168942,457,0,0.171682,0.230640
4,4,33570,2107,31463,0.062764,0.010170,152,170388,442,0,0.164593,0.255892


## 10. Empty-code guard and static-feature alignment


In [12]:
# ============================================================================
# Repair rare empty abstracted representations and align static features.
# ============================================================================

CODE_COLUMN = "abstracted_code_v1"
EMPTY_ABSTRACTED_SENTINEL = "EMPTY_ABSTRACTED_CODE_SAMPLE"

for frame_name, frame in [
    ("development", dev_df),
    ("outer_holdout_locked", holdout_df),
]:
    if CODE_COLUMN not in frame.columns:
        raise KeyError(
            f"{frame_name} frame is missing {CODE_COLUMN}.\n"
            f"Available columns: {sorted(frame.columns.tolist())}"
        )

    frame[CODE_COLUMN] = frame[CODE_COLUMN].fillna("").astype(str)
    empty_mask = frame[CODE_COLUMN].str.strip().eq("")
    n_empty = int(empty_mask.sum())
    print(f"Empty {CODE_COLUMN} rows in {frame_name}:", n_empty)

    if n_empty:
        frame.loc[empty_mask, CODE_COLUMN] = EMPTY_ABSTRACTED_SENTINEL
        print(
            f"Replaced {n_empty} empty rows in {frame_name} with "
            f"{EMPTY_ABSTRACTED_SENTINEL}"
        )

assert not dev_df[CODE_COLUMN].str.strip().eq("").any()

expected_static_columns = [
    "source_row_id",
    *static_features.FEATURE_COLUMNS,
]
if list(static_df.columns) != expected_static_columns:
    raise ValueError(
        "Static-cache schema mismatch.\n"
        f"Expected: {expected_static_columns}\n"
        f"Actual: {list(static_df.columns)}"
    )

if static_df["source_row_id"].duplicated().any():
    raise ValueError("Static cache contains duplicate source_row_id values.")

static_indexed = static_df.set_index("source_row_id", drop=False).sort_index()
static_ids = set(static_indexed.index.tolist())

if not dev_ids.issubset(static_ids):
    raise ValueError("Static cache misses development IDs.")
if not holdout_ids.issubset(static_ids):
    raise ValueError("Static cache misses locked outer-holdout IDs.")

dev_static_df = static_indexed.loc[development_ids_sorted].copy().reset_index(drop=True)
holdout_static_df = static_indexed.loc[holdout_ids_sorted].copy().reset_index(drop=True)

if set(dev_static_df["source_row_id"]) != set(dev_df["source_row_id"]):
    raise RuntimeError("Development static features do not match dev_df IDs.")

print("Preprocessing guard passed.")
print("Development static feature rows:", f"{len(dev_static_df):,}")
print("Static features:", len(static_features.FEATURE_COLUMNS))


Empty abstracted_code_v1 rows in development: 1
Replaced 1 empty rows in development with EMPTY_ABSTRACTED_CODE_SAMPLE
Empty abstracted_code_v1 rows in outer_holdout_locked: 0
Preprocessing guard passed.
Development static feature rows: 203,958
Static features: 54


## 11. Configure EXP-2-IDABS MLP

This keeps the same EXP-2 pipeline shape as the normalized-code MLP and only changes the text column to `abstracted_code_v1`.


In [13]:
# ============================================================================
# 11. Configure EXP-2-IDABS MLP
# ============================================================================
#
# Important:
# The EXP-2 runner in your repository is a custom PyTorch-based module, not
# sklearn's MLPClassifier. Therefore, we should not pass sklearn-style arguments
# such as `mlp_hidden_layer_sizes` unless the current Exp2Config actually
# supports them.
#
# This cell builds the configuration defensively:
# 1. Define the IDABS experiment settings we want.
# 2. Keep only arguments accepted by the current exp2_mlp.Exp2Config dataclass.
# 3. Print ignored arguments so configuration mismatches are visible.
#
# The mandatory change for EXP-2-IDABS is:
#     code_column = "abstracted_code_v1"
#
# All other unsupported fields fall back to the official defaults already
# defined inside case_study_1.exp2.exp2_mlp.Exp2Config.

from dataclasses import fields

supported_exp2_fields = {field.name for field in fields(exp2_mlp.Exp2Config)}

candidate_exp2_kwargs = {
    # Experiment identity and dataset columns.
    "experiment_name": "cs1_exp2_mlp_idabs_inner_dev_grouped",
    "code_column": "abstracted_code_v1",
    "source_id_column": "source_row_id",
    "label_column": "label",
    "project_column": "project",
    "fold_column": "fold",
    "n_splits": INNER_N_SPLITS,
    "random_state": MODEL_RANDOM_STATE,
    "decision_threshold": 0.50,

    # Lexical representation: same budget family as EXP-0/EXP-1/EXP-2.
    "word_ngram_range": (1, 3),
    "word_min_df": 3,
    "word_max_df": 0.995,
    "word_max_features": 50_000,
    "char_analyzer": "char",
    "char_ngram_range": (3, 4),
    "char_min_df": 8,
    "char_max_df": 0.995,
    "char_max_features": 60_000,
    "lowercase": False,
    "sublinear_tf": True,
    "tfidf_norm": "l2",

    # Train-fold-only dense compression.
    "svd_n_components": 256,
    "svd_algorithm": "randomized",
    "svd_n_iter": 5,
    "svd_n_oversamples": 10,

    # Common names used by custom PyTorch MLP configs.
    # Only the names supported by the current Exp2Config will be used.
    "hidden_dims": (128, 64),
    "hidden_layer_sizes": (128, 64),
    "hidden_units": (128, 64),
    "dropout": 0.20,
    "learning_rate": 1e-3,
    "lr": 1e-3,
    "weight_decay": 1e-4,
    "batch_size": 512,
    "max_epochs": 35,
    "epochs": 35,
    "patience": 4,
    "early_stopping_patience": 4,
    "validation_fraction": 0.15,
    "val_fraction": 0.15,
    "device": "cuda" if torch.cuda.is_available() else "cpu",

    # Logging.
    "verbose": True,
}

exp2_kwargs = {
    key: value
    for key, value in candidate_exp2_kwargs.items()
    if key in supported_exp2_fields
}

ignored_exp2_kwargs = sorted(
    key for key in candidate_exp2_kwargs
    if key not in supported_exp2_fields
)

if "code_column" not in exp2_kwargs:
    raise RuntimeError(
        "Current exp2_mlp.Exp2Config does not expose code_column. "
        "Cannot run an IDABS representation experiment safely."
    )

exp2_config = exp2_mlp.Exp2Config(**exp2_kwargs)

def _config_value(config, names, default="not exposed by current Exp2Config"):
    for name in names:
        if hasattr(config, name):
            return getattr(config, name)
    return default

print("EXP-2-IDABS configuration")
print("=" * 80)
print("Experiment name:", exp2_config.experiment_name)
print("Input column:", exp2_config.code_column)
print("Word max features:", _config_value(exp2_config, ["word_max_features"]))
print("Char max features:", _config_value(exp2_config, ["char_max_features"]))
print("SVD components:", _config_value(exp2_config, ["svd_n_components"]))
print(
    "MLP hidden layers:",
    _config_value(
        exp2_config,
        ["hidden_dims", "hidden_layer_sizes", "hidden_units", "mlp_hidden_layer_sizes"],
    ),
)
print("Learning rate:", _config_value(exp2_config, ["learning_rate", "lr"]))
print("Batch size:", _config_value(exp2_config, ["batch_size"]))
print("Max epochs:", _config_value(exp2_config, ["max_epochs", "epochs"]))
print("Device:", _config_value(exp2_config, ["device"]))
print("Model random state:", MODEL_RANDOM_STATE)
print("Output directory:", EXP2_IDABS_OUTPUT_DIR)
print("Final outer holdout: locked and untouched")

print("\nAccepted Exp2Config fields used:")
print(sorted(exp2_kwargs.keys()))

print("\nCandidate settings ignored because this Exp2Config does not define them:")
print(ignored_exp2_kwargs)

print("\nAll supported Exp2Config fields in this repository:")
print(sorted(supported_exp2_fields))


EXP-2-IDABS configuration
Experiment name: cs1_exp2_mlp_idabs_inner_dev_grouped
Input column: abstracted_code_v1
Word max features: 50000
Char max features: 60000
SVD components: 256
MLP hidden layers: not exposed by current Exp2Config
Learning rate: 0.001
Batch size: not exposed by current Exp2Config
Max epochs: 35
Device: cuda
Model random state: 42
Output directory: /content/drive/MyDrive/IntelligentSystemProject/VulnerabilityDetectionData/outputs/cs1_project_holdout20_innercv_v1/exp2_mlp_idabs_dev_cv_v1
Final outer holdout: locked and untouched

Accepted Exp2Config fields used:
['char_analyzer', 'char_max_df', 'char_max_features', 'char_min_df', 'char_ngram_range', 'code_column', 'decision_threshold', 'device', 'early_stopping_patience', 'experiment_name', 'fold_column', 'label_column', 'learning_rate', 'lowercase', 'max_epochs', 'n_splits', 'project_column', 'random_state', 'source_id_column', 'sublinear_tf', 'svd_algorithm', 'svd_n_components', 'svd_n_iter', 'svd_n_oversamples', 

## 12. Optional profile fold

This runs one fold only. It is a smoke test and computational profile, not the official result.


In [14]:
PROFILE_FOLD_ID = int(
    inner_fold_summary.loc[
        inner_fold_summary["train_rows"].idxmax(),
        "fold",
    ]
)
profile_outer_train_rows = int(
    inner_fold_summary.loc[
        inner_fold_summary["fold"] == PROFILE_FOLD_ID,
        "train_rows",
    ].iloc[0]
)

print(
    "Selected profile fold:",
    f"Fold {PROFILE_FOLD_ID} with {profile_outer_train_rows:,} outer-train rows."
)

if RUN_PROFILE_FOLD:
    profile_start = time.perf_counter()
    exp2_profile_results = exp2_mlp.run_exp2_profile_fold(
        normalized_frame=dev_df,
        static_features_frame=dev_static_df,
        manifest=inner_manifest_df,
        fold_id=PROFILE_FOLD_ID,
        config=exp2_config,
    )
    profile_seconds = time.perf_counter() - profile_start

    print("\nProfile run completed in %.2f minutes." % (profile_seconds / 60.0))

    EXP2_IDABS_PROFILE_DIR.mkdir(parents=True, exist_ok=True)
    exp2_profile_results["predictions"].to_csv(
        EXP2_IDABS_PROFILE_DIR / "profile_fold_predictions.csv",
        index=False,
    )
    exp2_profile_results["training_history"].to_csv(
        EXP2_IDABS_PROFILE_DIR / "profile_training_history.csv",
        index=False,
    )
    exp2_profile_results["training_metadata"].to_csv(
        EXP2_IDABS_PROFILE_DIR / "profile_training_metadata.csv",
        index=False,
    )
    with (EXP2_IDABS_PROFILE_DIR / "profile_metrics.json").open("w", encoding="utf-8") as file:
        json.dump(exp2_profile_results["profile_metrics"], file, indent=2)
    with (EXP2_IDABS_PROFILE_DIR / "profile_validation_metrics.json").open("w", encoding="utf-8") as file:
        json.dump(exp2_profile_results["validation_metrics"], file, indent=2)
    with (EXP2_IDABS_PROFILE_DIR / "profile_default_threshold_metrics.json").open("w", encoding="utf-8") as file:
        json.dump(exp2_profile_results["default_threshold_metrics"], file, indent=2)

    print("Profile artifacts saved to:", EXP2_IDABS_PROFILE_DIR)
else:
    print("RUN_PROFILE_FOLD=False; profile skipped.")


Selected profile fold: Fold 4 with 170,388 outer-train rows.
RUN_PROFILE_FOLD=False; profile skipped.


## 13. Inspect profile result


In [15]:
import matplotlib.pyplot as plt

if "exp2_profile_results" not in globals():
    print("No profile result in memory. Run the profile cell first or skip this section.")
else:
    print("Largest-fold computational and training profile:")
    display(exp2_profile_results["training_metadata"])

    selected_threshold = float(
        exp2_profile_results["predictions"]["selected_validation_threshold"].iloc[0]
    )
    print("\nThreshold strategy: inner project-validation F1 selection.")
    print(f"Selected validation threshold: {selected_threshold:.3f}")

    print("\nInner-validation metrics used for threshold/checkpoint context:")
    display(
        pd.DataFrame(
            list(exp2_profile_results["validation_metrics"].items()),
            columns=["metric", "value"],
        )
    )

    print("\nOuter held-out profile-fold metrics — descriptive only, not official full CV:")
    display(
        pd.DataFrame(
            list(exp2_profile_results["profile_metrics"].items()),
            columns=["metric", "value"],
        )
    )

    print("\nFixed threshold = 0.50 diagnostic only:")
    display(
        pd.DataFrame(
            list(exp2_profile_results["default_threshold_metrics"].items()),
            columns=["metric", "value"],
        )
    )

    history_df = exp2_profile_results["training_history"].copy()
    print("\nEpoch-level training history:")
    display(history_df)

    plt.figure(figsize=(7, 4))
    plt.plot(history_df["epoch"], history_df["validation_pr_auc"], marker="o")
    plt.xlabel("Epoch")
    plt.ylabel("Validation PR-AUC")
    plt.title("EXP-2-IDABS profile: validation PR-AUC by epoch")
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(7, 4))
    plt.plot(history_df["epoch"], history_df["train_weighted_bce_loss"], label="train")
    plt.plot(history_df["epoch"], history_df["validation_weighted_bce_loss"], label="validation")
    plt.xlabel("Epoch")
    plt.ylabel("Weighted BCE loss")
    plt.title("EXP-2-IDABS profile: weighted loss by epoch")
    plt.legend()
    plt.tight_layout()
    plt.show()


No profile result in memory. Run the profile cell first or skip this section.


## 14. Official five-fold development CV

This is the official EXP-2-IDABS development-only result. The frozen outer holdout is not used.


In [16]:
if RUN_OFFICIAL_DEV_CV:
    if REQUIRE_GPU_FOR_OFFICIAL_RUN and not torch.cuda.is_available():
        raise RuntimeError(
            "GPU is required for this official MLP run. In Colab select "
            "Runtime > Change runtime type > GPU, then restart and rerun setup cells."
        )

    existing_files = [p for p in EXP2_IDABS_OUTPUT_DIR.iterdir() if p.name != "profile_fold"]
    if existing_files:
        raise RuntimeError(
            "EXP-2-IDABS official output directory already contains files.\n"
            f"Directory: {EXP2_IDABS_OUTPUT_DIR}\n"
            "Do not mix reruns. Change RUN_VERSION, for example RUN_VERSION='v2', "
            "and rerun the setup cells."
        )

    official_start = time.perf_counter()
    exp2_idabs_results = exp2_mlp.run_exp2(
        normalized_frame=dev_df,
        static_features_frame=dev_static_df,
        manifest=inner_manifest_df,
        config=exp2_config,
        output_dir=EXP2_IDABS_OUTPUT_DIR,
        additional_metadata={
            "run_kind": "exp2_identifier_abstraction_development_cv",
            "evaluation_stage": "inner_development_cv_only",
            "global_outer_holdout_used": False,
            "input_parquet": str(ABSTRACTED_DATA_PATH),
            "input_column": "abstracted_code_v1",
            "outer_holdout_manifest_path": str(OUTER_MANIFEST_PATH),
            "inner_manifest_path": str(INNER_MANIFEST_PATH),
            "static_feature_cache_path": str(STATIC_FEATURE_PATH),
            "empty_abstracted_code_repair_token": EMPTY_ABSTRACTED_SENTINEL,
            "repo_commit": repo_commit,
            "note": (
                "Supplementary IDABS representation study. Development CV only; "
                "does not reopen final holdout model selection."
            ),
        },
    )
    official_seconds = time.perf_counter() - official_start
    print("\nOfficial EXP-2-IDABS development CV completed in %.2f minutes." % (official_seconds / 60.0))
    print("Artifacts saved to:", EXP2_IDABS_OUTPUT_DIR)
else:
    print("RUN_OFFICIAL_DEV_CV=False; official five-fold CV skipped.")


[12:58:54] CS1-EXP2 official run started: 5-fold grouped CV.
[12:58:54] Configuration: lexical<= 110,000, SVD=256, static=54, MLP=128->64->1, device=cuda.
[12:58:55] Fold 1/5 started | outer-train=148,449, outer-test=55,509, outer-train projects=593, outer-test projects=1.
[12:59:32] Fold 1/5 | inner project split: fit=118,745 rows / 472 projects; validation=29,704 rows / 121 projects.
[12:59:32] Fold 1/5 | fitting inner-train word TF-IDF...
[13:00:15] Fold 1/5 | word TF-IDF done in 42.8s (50,000 features).
[13:00:15] Fold 1/5 | fitting inner-train character TF-IDF...
[13:02:05] Fold 1/5 | character TF-IDF done in 1.84 min (60,000 features).
[13:02:05] Fold 1/5 | joining sparse lexical matrices...
[13:02:06] Fold 1/5 | fitting inner-train TruncatedSVD (256 components)...
[13:04:01] Fold 1/5 | TruncatedSVD done in 1.91 min (explained variance ratio sum=0.3531).
[13:04:01] Fold 1/5 | loading deterministic static features...
[13:04:01] Fold 1/5 | fitting inner-train StandardScaler...
[13:

## 15. Load existing official artifacts after restart

Use this section when Colab restarts after a completed run.


In [17]:
# ============================================================================
# The EXP-2 module names artifacts using the experiment name. This loader uses
# flexible glob patterns so it keeps working even if small filename details differ.
# ============================================================================

artifact_patterns = {
    "pooled_metrics": "*pooled_metrics.json",
    "validation_operating_pooled_metrics": "*validation*pooled_metrics.json",
    "fold_metrics": "*fold_metrics.csv",
    "fold_summary": "*fold_summary.csv",
    "fold_training": "*fold_training*.csv",
    "training_history": "*training_history*.csv",
    "oof_predictions": "*oof_predictions*.parquet",
}

loaded_exp2_idabs_artifacts = {}

if LOAD_EXISTING_DEV_CV_ARTIFACTS:
    print("Searching artifacts in:", EXP2_IDABS_OUTPUT_DIR)
    for key, pattern in artifact_patterns.items():
        matches = sorted(EXP2_IDABS_OUTPUT_DIR.glob(pattern))
        print(f"{key}: {[p.name for p in matches]}")

    # Load JSON metrics. Prefer non-profile artifacts in the official output dir.
    json_candidates = sorted(EXP2_IDABS_OUTPUT_DIR.glob("*.json"))
    for path in json_candidates:
        name = path.name
        if "profile" in name:
            continue
        if "pooled_metrics" in name or "metrics" in name:
            try:
                with path.open("r", encoding="utf-8") as file:
                    loaded_exp2_idabs_artifacts[name] = json.load(file)
            except Exception as exc:
                print("Could not load JSON:", path, exc)

    # Load common CSV tables.
    for path in sorted(EXP2_IDABS_OUTPUT_DIR.glob("*.csv")):
        name = path.name
        if "profile" in name:
            continue
        try:
            loaded_exp2_idabs_artifacts[name] = pd.read_csv(path)
        except Exception as exc:
            print("Could not load CSV:", path, exc)

    print("Loaded artifact keys:", sorted(loaded_exp2_idabs_artifacts.keys()))
else:
    print("LOAD_EXISTING_DEV_CV_ARTIFACTS=False; artifact loading skipped.")


Searching artifacts in: /content/drive/MyDrive/IntelligentSystemProject/VulnerabilityDetectionData/outputs/cs1_project_holdout20_innercv_v1/exp2_mlp_idabs_dev_cv_v1
pooled_metrics: ['cs1_exp2_mlp_idabs_inner_dev_grouped_pooled_metrics.json', 'cs1_exp2_mlp_idabs_inner_dev_grouped_validation_operating_pooled_metrics.json']
validation_operating_pooled_metrics: ['cs1_exp2_mlp_idabs_inner_dev_grouped_validation_operating_pooled_metrics.json']
fold_metrics: ['cs1_exp2_mlp_idabs_inner_dev_grouped_fold_metrics.csv', 'cs1_exp2_mlp_idabs_inner_dev_grouped_validation_operating_fold_metrics.csv']
fold_summary: ['cs1_exp2_mlp_idabs_inner_dev_grouped_fold_summary.csv', 'cs1_exp2_mlp_idabs_inner_dev_grouped_validation_operating_fold_summary.csv']
fold_training: ['cs1_exp2_mlp_idabs_inner_dev_grouped_fold_training.csv']
training_history: ['cs1_exp2_mlp_idabs_inner_dev_grouped_training_history.csv']
oof_predictions: ['cs1_exp2_mlp_idabs_inner_dev_grouped_oof_predictions.parquet']
Loaded artifact keys: 

## 16. Review official EXP-2-IDABS result


In [18]:
def _metrics_to_df(metrics, ordered_keys=None):
    if ordered_keys is None:
        ordered_keys = list(metrics.keys())
    return pd.DataFrame(
        [
            {"metric": key, "value": metrics.get(key)}
            for key in ordered_keys
            if key in metrics
        ]
    )

standard_evaluation = None
validation_operating = None
fold_training_df = None
training_history_df = None

if "exp2_idabs_results" in globals():
    standard_evaluation = exp2_idabs_results.get("evaluation")
    validation_operating = exp2_idabs_results.get("validation_operating_evaluation")
    fold_training_df = exp2_idabs_results.get("fold_training")
    training_history_df = exp2_idabs_results.get("training_history")
else:
    # Best-effort reconstruction from loaded artifacts.
    json_items = {
        key: value
        for key, value in loaded_exp2_idabs_artifacts.items()
        if isinstance(value, dict)
    }
    csv_items = {
        key: value
        for key, value in loaded_exp2_idabs_artifacts.items()
        if isinstance(value, pd.DataFrame)
    }

    # Locate metric JSON files.
    pooled_candidates = [
        (name, data)
        for name, data in json_items.items()
        if "pooled_metrics" in name and "validation" not in name
    ]
    validation_candidates = [
        (name, data)
        for name, data in json_items.items()
        if "validation" in name and "pooled_metrics" in name
    ]

    if pooled_candidates:
        standard_evaluation = {"pooled_metrics": pooled_candidates[0][1]}
        print("Loaded standard pooled metrics from:", pooled_candidates[0][0])
    if validation_candidates:
        validation_operating = {"pooled_metrics": validation_candidates[0][1]}
        print("Loaded validation-operating metrics from:", validation_candidates[0][0])

    for name, df in csv_items.items():
        if "fold_training" in name:
            fold_training_df = df
        elif "training_history" in name:
            training_history_df = df

if standard_evaluation is None:
    print(
        "No official EXP-2-IDABS result found in memory or artifacts. "
        "Run Section 14 with RUN_OFFICIAL_DEV_CV=True."
    )
else:
    ranking_keys = [
        "n_samples",
        "vulnerable_1",
        "non_vulnerable_0",
        "positive_rate",
        "average_precision_pr_auc",
    ]
    print("EXP-2-IDABS pooled OOF ranking metrics — PR-AUC is primary:")
    display(_metrics_to_df(standard_evaluation["pooled_metrics"], ranking_keys))

    if validation_operating is not None:
        operating_keys = [
            "n_samples", "vulnerable_1", "non_vulnerable_0", "positive_rate",
            "average_precision_pr_auc", "precision", "recall", "f1", "mcc",
            "specificity", "false_positive_rate", "false_negative_rate",
            "true_negative", "false_positive", "false_negative", "true_positive",
            "predicted_positive", "predicted_positive_rate", "threshold_strategy",
            "selected_threshold_min", "selected_threshold_max", "selected_threshold_mean",
        ]
        print("\nEXP-2-IDABS pooled operating metrics — thresholds selected on inner validation only:")
        display(_metrics_to_df(validation_operating["pooled_metrics"], operating_keys))

    if fold_training_df is not None:
        print("\nFold-level preprocessing, training, and threshold information:")
        fold_columns = [
            "fold", "outer_train_rows", "outer_test_rows",
            "inner_fit_rows", "inner_validation_rows",
            "outer_train_unique_projects", "outer_test_unique_projects",
            "inner_fit_projects", "inner_validation_projects",
            "device", "final_dense_features", "best_epoch", "epochs_completed",
            "selected_validation_threshold", "validation_average_precision_pr_auc",
            "validation_f1", "mlp_training_seconds", "total_fold_seconds",
            "peak_gpu_memory_bytes",
        ]
        available_columns = [c for c in fold_columns if c in fold_training_df.columns]
        display(fold_training_df[available_columns])

    if training_history_df is not None:
        print("\nEpoch-level training history:")
        display(training_history_df)


EXP-2-IDABS pooled OOF ranking metrics — PR-AUC is primary:


,metric,value
0,n_samples,203958.000000
1,vulnerable_1,10727.000000
2,non_vulnerable_0,193231.000000
3,positive_rate,0.052594
4,average_precision_pr_auc,0.137544



EXP-2-IDABS pooled operating metrics — thresholds selected on inner validation only:


,metric,value
0,n_samples,203958
1,vulnerable_1,10727
2,non_vulnerable_0,193231
3,positive_rate,0.052594
4,average_precision_pr_auc,0.137544
5,precision,0.133638
6,recall,0.413909
7,f1,0.202043
8,mcc,0.160159
9,specificity,0.851038



Fold-level preprocessing, training, and threshold information:


,fold,outer_train_rows,outer_test_rows,inner_fit_rows,inner_validation_rows,outer_train_unique_projects,outer_test_unique_projects,inner_fit_projects,inner_validation_projects,device,final_dense_features,best_epoch,epochs_completed,selected_validation_threshold,validation_average_precision_pr_auc,validation_f1,mlp_training_seconds,total_fold_seconds,peak_gpu_memory_bytes
0,0,148449,55509,118745,29704,593,1,472,121,cuda,310,1,5,0.65,0.171368,0.242686,8.663487,322.983250,26257920
1,1,163167,40791,130587,32580,436,158,326,110,cuda,310,1,5,0.70,0.123845,0.199247,8.104291,334.208788,26254848
2,2,164886,39072,131895,32991,448,146,335,113,cuda,310,1,5,0.65,0.130446,0.190764,7.599086,337.188322,24759296
3,3,168942,35016,135151,33791,457,137,359,98,cuda,310,1,5,0.61,0.136717,0.210297,8.273278,328.640245,26257920
4,4,170388,33570,136295,34093,442,152,337,105,cuda,310,2,6,0.64,0.165854,0.253298,11.132545,331.763151,25108992



Epoch-level training history:


,fold,epoch,train_weighted_bce_loss,validation_weighted_bce_loss,validation_pr_auc,learning_rate,is_best_epoch
0,0,1,1.144825,1.341094,0.171368,0.001,True
1,0,2,1.041347,1.363373,0.165602,0.001,False
2,0,3,0.988361,1.390776,0.161978,0.001,False
3,0,4,0.921395,1.464266,0.157440,0.001,False
4,0,5,0.836473,1.599917,0.137539,0.001,False
5,1,1,1.154585,1.042459,0.123845,0.001,True
6,1,2,1.065927,1.055685,0.122131,0.001,False
7,1,3,1.012562,1.068284,0.117746,0.001,False
8,1,4,0.939706,1.096458,0.115465,0.001,False
9,1,5,0.850129,1.161303,0.101015,0.001,False


## 17. Development-only comparison table

This table is for interpretation only. It does not use the frozen outer holdout.


In [20]:
import pandas as pd

main_model_comparison = pd.DataFrame([
    {
        "experiment": "EXP-0 fixed LR",
        "representation": "normalized_code",
        "scope": "development pooled OOF",
        "pr_auc": 0.125205,
        "role": "original fixed baseline",
    },
    {
        "experiment": "EXP-1 RF",
        "representation": "SVD(TF-IDF normalized_code)+static",
        "scope": "development pooled OOF",
        "pr_auc": 0.124803,
        "role": "original fixed RF pipeline",
    },
    {
        "experiment": "EXP-2 MLP",
        "representation": "SVD(TF-IDF normalized_code)+static",
        "scope": "development pooled OOF",
        "pr_auc": 0.138157,
        "role": "selected original final model",
    },
]).sort_values("pr_auc", ascending=False)

exp0_tuning_comparison = pd.DataFrame([
    {
        "experiment": "EXP-0 fixed LR",
        "representation": "normalized_code",
        "scope": "development pooled OOF",
        "pr_auc": 0.125205,
        "role": "fixed baseline",
    },
    {
        "experiment": "EXP-0 nested-alpha LR",
        "representation": "normalized_code",
        "scope": "development nested pooled OOF",
        "pr_auc": 0.141971,
        "role": "supplementary tuned LR",
    },
    {
        "experiment": "EXP-0-IDABS nested-alpha LR",
        "representation": "abstracted_code_v1",
        "scope": "development nested pooled OOF",
        "pr_auc": 0.136888,
        "role": "supplementary tuned LR with IDABS",
    },
]).sort_values("pr_auc", ascending=False)

representation_comparison = pd.DataFrame([
    {
        "model_family": "EXP-0 nested-alpha LR",
        "normalized_pr_auc": 0.141971,
        "idabs_pr_auc": 0.136888,
        "delta_idabs_minus_normalized": 0.136888 - 0.141971,
    },
    {
        "model_family": "EXP-1 RF",
        "normalized_pr_auc": 0.124803,
        "idabs_pr_auc": 0.124013,
        "delta_idabs_minus_normalized": 0.124013 - 0.124803,
    },
    {
        "model_family": "EXP-2 MLP",
        "normalized_pr_auc": 0.138157,
        "idabs_pr_auc": 0.137544,
        "delta_idabs_minus_normalized": 0.137544 - 0.138157,
    },
])

display(main_model_comparison)
display(exp0_tuning_comparison)
display(representation_comparison)

,experiment,representation,scope,pr_auc,role
2,EXP-2 MLP,SVD(TF-IDF normalized_code)+static,development pooled OOF,0.138157,selected original final model
0,EXP-0 fixed LR,normalized_code,development pooled OOF,0.125205,original fixed baseline
1,EXP-1 RF,SVD(TF-IDF normalized_code)+static,development pooled OOF,0.124803,original fixed RF pipeline


,experiment,representation,scope,pr_auc,role
1,EXP-0 nested-alpha LR,normalized_code,development nested pooled OOF,0.141971,supplementary tuned LR
2,EXP-0-IDABS nested-alpha LR,abstracted_code_v1,development nested pooled OOF,0.136888,supplementary tuned LR with IDABS
0,EXP-0 fixed LR,normalized_code,development pooled OOF,0.125205,fixed baseline


,model_family,normalized_pr_auc,idabs_pr_auc,delta_idabs_minus_normalized
0,EXP-0 nested-alpha LR,0.141971,0.136888,-0.005083
1,EXP-1 RF,0.124803,0.124013,-0.000790
2,EXP-2 MLP,0.138157,0.137544,-0.000613


## 18. Final outer holdout remains locked

This cell intentionally blocks outer-holdout evaluation for EXP-2-IDABS.

The frozen 20% holdout has already been consumed once for the locked selected EXP-2 normalized-code model. EXP-2-IDABS is a supplementary representation study and must remain development-only.


In [ ]:
if RUN_FINAL_OUTER_HOLDOUT:
    raise RuntimeError(
        "Outer-holdout evaluation for EXP-2-IDABS is intentionally disabled.\n\n"
        "Reason:\n"
        "- EXP-2-IDABS is a supplementary representation experiment created after "
        "the final selected EXP-2 normalized-code holdout result was inspected.\n"
        "- Running this new candidate on the same frozen 20% holdout would make it "
        "a post-hoc test-set comparison.\n\n"
        "Use only the development-CV result for EXP-2-IDABS."
    )

print("Final outer holdout remains locked. EXP-2-IDABS is development-only.")


## 19. Report note

Use this wording after the official run completes:

> EXP-2-IDABS was evaluated as a supplementary representation experiment by replacing `normalized_code` with `abstracted_code_v1` while preserving the EXP-2 MLP pipeline. The model was evaluated only on the frozen 80% development partition using the same project-grouped five-fold protocol. The global 20% outer holdout was not used for this post-hoc representation study.
